In [1]:
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive/NU_NLP_project/project_dataset"

dev_test  parallel-n


In [3]:
import pandas as pd

# Define the file paths
english_file = "/content/drive/MyDrive/NU_NLP_project/project_dataset/parallel-n/IITB.en-hi.en"
hindi_file = "/content/drive/MyDrive/NU_NLP_project/project_dataset/parallel-n/IITB.en-hi.hi"

# Load the English and Hindi data
with open(english_file, 'r', encoding='utf-8') as en_file:
    english_sentences = en_file.readlines()

with open(hindi_file, 'r', encoding='utf-8') as hi_file:
    hindi_sentences = hi_file.readlines()

# Combine into a pandas DataFrame
data = pd.DataFrame({
    'English': [sentence.strip() for sentence in english_sentences],
    'Hindi': [sentence.strip() for sentence in hindi_sentences]
})

# Check the first few rows
print(data.head(10))

# Save to a CSV for easier exploration (optional)
data.to_csv("combined_dataset.csv", index=False, encoding='utf-8')

                                             English  \
0     Give your application an accessibility workout   
1                  Accerciser Accessibility Explorer   
2     The default plugin layout for the bottom panel   
3        The default plugin layout for the top panel   
4     A list of plugins that are disabled by default   
5                                 Highlight duration   
6  The duration of the highlight box when selecti...   
7                             Highlight border color   
8     The color and opacity of the highlight border.   
9                               Highlight fill color   

                                               Hindi  
0    अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें  
1                    एक्सेर्साइसर पहुंचनीयता अन्वेषक  
2              निचले पटल के लिए डिफोल्ट प्लग-इन खाका  
3               ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका  
4  उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...  
5                               अवधि को हाइलाइट रकें 

In [4]:
# File paths for validation and test sets
dev_english_file = "/content/drive/MyDrive/NU_NLP_project/project_dataset/dev_test/dev.en"
dev_hindi_file = "/content/drive/MyDrive/NU_NLP_project/project_dataset/dev_test/dev.hi"
test_english_file = "/content/drive/MyDrive/NU_NLP_project/project_dataset/dev_test/test.en"
test_hindi_file = "/content/drive/MyDrive/NU_NLP_project/project_dataset/dev_test/test.hi"

# Load validation set
with open(dev_english_file, 'r', encoding='utf-8') as dev_en_file, \
     open(dev_hindi_file, 'r', encoding='utf-8') as dev_hi_file:
    val_data = pd.DataFrame({
        'English': [line.strip() for line in dev_en_file.readlines()],
        'Hindi': [line.strip() for line in dev_hi_file.readlines()]
    })

# Load test set
with open(test_english_file, 'r', encoding='utf-8') as test_en_file, \
     open(test_hindi_file, 'r', encoding='utf-8') as test_hi_file:
    test_data = pd.DataFrame({
        'English': [line.strip() for line in test_en_file.readlines()],
        'Hindi': [line.strip() for line in test_hi_file.readlines()]
    })

# Display first few rows of validation and test sets
print("Validation Data Sample:")
print(val_data.head())

print("Test Data Sample:")
print(test_data.head())

Validation Data Sample:
                                             English  \
0  Students of the Dattatreya city Municipal corp...   
1  With encouragement from Principal Sandhya Medp...   
2  Rajesh Gavre, the President of the MNPA teache...   
3                 Ramesh Saatpute examined the fort.   
4  Students like Nikhil Kavle, Darshan Gedekar, S...   

                                               Hindi  
0  महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स...  
1  प्रधानाध्यापक संध्या मेडपल्लीवार के प्रोत्साहि...  
2  मनपा शिक्षक संघ के अध्यक्ष राजेश गवरे ने स्कूल...  
3              किले का परीक्षण रमेश सातपुते ने किया।  
4  किला निर्माण में निखिल कावले, दर्शन गेड़ेकर, स...  
Test Data Sample:
                                             English  \
0                           A black box in your car?   
1  As America's road planners struggle to find th...   
2  The devices, which track every mile a motorist...   
3  The usually dull arena of highway planning has...   
4  Libertar

In [5]:
!pip install transformers sentencepiece torch accelerate

In [7]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `colab`


In [8]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")
model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b")

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [8]:
# Move the model to the GPU
model = model.to("cuda")
input_text = "Translate to Hindi: How are you?"
# Tokenize the input
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")  # Ensure inputs are on the GPU

# Generate output
outputs = model.generate(**inputs, max_length=50)

# Decode and print the output
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Output:", decoded_output)

The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.


Generated Output: Translate to Hindi: How are you?

Translate to Hindi: How are you?

Translate to Hindi: How are you?

Translate to Hindi: How are you?

Translate to Hindi: How are you?

Translate to Hindi:


In [9]:
import torch

print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Number of GPUs available
print(torch.cuda.current_device())  # Current active GPU index
print(torch.cuda.get_device_name(0))  # Name of the GPU

True
1
0
Tesla T4


In [10]:
def tokenize_function(example):
    return tokenizer(
        example['English'],
        text_target=example['Hindi'],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [11]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [12]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(data)
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1659083 [00:00<?, ? examples/s]

In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/NU_NLP_project/results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/content/drive/MyDrive/NU_NLP_project/logs",
    logging_steps=10,
    fp16=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)



/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Replace with validation dataset
    tokenizer=tokenizer,
    data_collator=data_collator
)



<ipython-input-14-e0739b4e9428>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


OutOfMemoryError: CUDA out of memory. Tried to allocate 42.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 29.06 MiB is free. Process 9997 has 14.72 GiB memory in use. Of the allocated memory 14.26 GiB is allocated by PyTorch, and 336.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)